In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration to display graphs in the notebook
%matplotlib inline
plt.style.use('ggplot')
sns.set(style='whitegrid')


# <b><span style='color:#F1A424'>|</span> Loading Multiple CSV Files <span style='color:#F1A424'>|</span></b>
Load several CSV files at once and review their basic information.


In [ ]:
# Load all CSV files into a dictionary of DataFrames
dataframes = {}

for csv_filename in ['Datasets/Tabular/Binary_pred/csv/dataset1_with_target.csv', 'Datasets/Tabular/Binary_pred/csv/dataset2_features_only.csv']:
    dataframes[csv_filename] = pd.read_csv(csv_filename)
    print(f"Table {csv_filename} shape: {dataframes[csv_filename].shape}")
    print(f"Column names: {dataframes[csv_filename].columns.tolist()}")

# Display the list of loaded DataFrames
list(dataframes.keys())


# <b><span style='color:#F1A424'>|</span> Merging DataFrames on Common ID <span style='color:#F1A424'>|</span></b>
Combine multiple tables into one by joining on a shared identifier column.


In [ ]:
# Merge all DataFrames on the common ID column
merged_df = None

for df_name, df in dataframes.items():
    if "ID" not in df.columns:
        print(f"Warning: The column 'ID' is not present in {df_name}, this file will be ignored")
        continue
        
    if merged_df is None:
        merged_df = df.copy()
    else:
        # Use a suffix to avoid column name duplication
        merged_df = pd.merge(merged_df, df, on="ID", how='outer', 
                            suffixes=('', f'_{df_name}'))
# Standardize merged df
df = merged_df

# Display information about the merged DataFrame
print("Final DataFrame shape:", df.shape)
print("Final DataFrame columns:", df.columns.tolist())
print("\nMerged DataFrame preview:")
df.head()


# <b><span style='color:#F1A424'>|</span> Categorical Data Encoding <span style='color:#F1A424'>|</span></b>
Convert categorical variables into numeric codes for analysis.


In [ ]:
# Transform categorical columns to numerical
from sklearn.preprocessing import LabelEncoder
label_encoders = {}
for col in df.select_dtypes(include=['object', 'category']).columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le
print("\nDataFrame after categorical encoding:")
df.head()


# <b><span style='color:#F1A424'>|</span> Correlation Analysis <span style='color:#F1A424'>|</span></b>
Analyze the relationships between features to inform feature selection.
We are using the absolute value of the correlation matrix to get insight on the magnitude of the correlation between the features, as the sign of the correlation is not important for the feature selection step.


In [ ]:
# Display correlation matrix
corr_matrix = df.corr().abs()
plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, cmap='inferno', vmin=0, vmax=1)
plt.title('Correlation Matrix')
plt.show()


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from torch.utils.data import TensorDataset, DataLoader


# <b><span style='color:#F1A424'>|</span> Data Preparation and Splitting <span style='color:#F1A424'>|</span></b>
Splitting the data into training and test sets, and normalizing the features.


In [ ]:
# Split and preprocess the data
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['YTarget']), df['YTarget'], test_size=0.2, random_state=42)

# Scale features for better model performance
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# <b><span style='color:#F1A424'>|</span> XGBoost Model Setup <span style='color:#F1A424'>|</span></b>
Initialization and configuration of the XGBoost model for multiclass classification.


In [ ]:
import xgboost as xgb

# Create DMatrix for XGBoost
dtrain = xgb.DMatrix(X_train_scaled, label=y_train)
dtest = xgb.DMatrix(X_test_scaled, label=y_test)

# Set parameters for XGBoost
params = {
    'objective': 'multi:softmax',
    'num_class': '2',
    'max_depth': 6,
    'eta': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 1,
    'eval_metric': 'mlogloss'
}


# <b><span style='color:#F1A424'>|</span> XGBoost Training & Evaluation <span style='color:#F1A424'>|</span></b>
Training the XGBoost model with early stopping and evaluating its performance.


In [ ]:
# Train the model with early stopping
num_rounds = 1000
eval_list = [(dtrain, 'train'), (dtest, 'eval')]
model = xgb.train(params, dtrain, num_rounds, eval_list,
               early_stopping_rounds=20, verbose_eval=100)

# Make predictions and evaluate
y_pred = model.predict(dtest)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


# <b><span style='color:#F1A424'>|</span> XGBoost Visualization <span style='color:#F1A424'>|</span></b>
Confusion matrix and feature importance for the XGBoost model.


In [ ]:
# Plot confusion matrix
plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

# Feature importance visualization
plt.figure(figsize=(14, 8))
importance = model.get_score(importance_type='gain')
features = list(importance.keys())
scores = list(importance.values())
indices = np.argsort(scores)[-20:]

colors = plt.cm.viridis(np.linspace(0, 0.9, len(indices)))
plt.barh(range(len(indices)), [scores[i] for i in indices], color=colors)
plt.yticks(range(len(indices)), [features[i] for i in indices])
plt.xlabel('Importance (gain)')
plt.ylabel('Features')
plt.title(f'Top {min(20, len(features))} Feature Importance')
plt.grid(axis='x', linestyle='--', alpha=0.6)

for i, v in enumerate([scores[idx] for idx in indices]):
    plt.text(v + 0.1, i, f"{v:.2f}", va='center')

plt.tight_layout()
plt.show()
